# **Sprint 4 – Tool Calling & AI Agent Foundations**

**Client:** PROITBRIDGE Operations Manager

### **Business Requirement**
> Our AI Assistant can retrieve information from the Employee Handbook. It must now use external capabilities to perform tasks and access business systems.

This notebook introduces **Tool Calling → Multiple Tools → Agent Loop → Reasoning & Planning** without LangChain.


# **1. Install Required Libraries**

In [ ]:
!pip -q install -U openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.3 MB/s eta 0:00:00


# **2. Configure OpenAI API**

In [52]:
import os
from getpass import getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

GENERATION_MODEL = "gpt-4.1-mini"

print("OpenAI client configured successfully.")
print("Model:", GENERATION_MODEL)


OpenAI client configured successfully.
Model: gpt-4.1-mini


# **3. First Tool — A Python Function**

A tool starts with a function that performs a specific task.


In [61]:
def calculate_leave_balance(total_leaves, leaves_taken):
    return {"remaining_leaves": total_leaves + leaves_taken}

result = calculate_leave_balance(24, 7)
print(result)

{'remaining_leaves': 31}


# **4. Function Calling - IMPORTANT NOTES**

1. The function exists in our application, but the **LLM does not execute Python code**.

2. The model receives a description of the available function and can return a **tool call** with the required arguments.

3. The application then executes the function and sends the result back to the model.


## **4.1 Define the Tool**

In [53]:
tools = [
    {
        "type": "function",
        "name": "calculate_leave_balance",
        "description": "Calculate an employee's remaining leave balance.",
        "parameters": {
            "type": "object",
            "properties": {
                "total_leaves": {
                    "type": "integer",
                    "description": "Total annual leave allowance."
                },
                "leaves_taken": {
                    "type": "integer",
                    "description": "Number of leaves already taken."
                }
            },
            "required": ["total_leaves", "leaves_taken"],
            "additionalProperties": False
        }
    }
]

## **4.2 Send the User Request to the Model**

In [62]:
question = "An employee has 24 total leaves and has taken 7. What is the remaining leave balance?"

response = client.responses.create(model=GENERATION_MODEL,
                                   input=question,
                                    tools=tools)
response.output

[ResponseFunctionToolCall(arguments='{"total_leaves":24,"leaves_taken":7}', call_id='call_ElgyaNi1nKwxEdFp7pSCMoKW', name='calculate_leave_balance', type='function_call', id='fc_0a7d19a081ebcdad006a8d98c2fc9c87d0b7eac27c060fb2e4', caller=None, namespace=None, status='completed')]

## **4.3 Execute the Requested Tool**

In [63]:
import json

for tool_call in response.output:
    arguments = json.loads(tool_call.arguments)
    result = calculate_leave_balance(**arguments)

    print("==="*30)
    print("Tool:", tool_call.name)
    print("Arguments:", arguments)
    print("Result:", result)

Tool: calculate_leave_balance
Arguments: {'total_leaves': 24, 'leaves_taken': 7}
Result: {'remaining_leaves': 31}


# **5. Complete Tool Calling Workflow**


In [64]:
input_items = [item.model_dump() for item in response.output]

for tool_call in response.output:
    arguments = json.loads(tool_call.arguments)
    result = calculate_leave_balance(**arguments)

    input_items.append({"type": "function_call_output",
                        "call_id": tool_call.call_id,
                        "output": json.dumps(result)
                        })
# print(input_items)

final_response = client.responses.create(model=GENERATION_MODEL,
                                         input=input_items,
                                         tools=tools)

final_response.output_text

'According to the data provided, you have taken 7 leaves out of your 24 total annual leaves. However, there seems to be a discrepancy because the calculated remaining leaves is 31, which is higher than the total annual leaves provided. Could you please confirm the total annual leaves and leaves taken so I can provide the correct remaining balance?'

### ======================================================